In [1]:
# 2. OpenAI 라이브러리 설치
!pip install openai

print("\n✅ 1단계 완료: 구글 드라이브 연결 및 라이브러리 설치 성공!")


✅ 1단계 완료: 구글 드라이브 연결 및 라이브러리 설치 성공!


In [3]:
# ===================================================================
# 셀 1: 기본 설정 및 라이브러리 로드
# ===================================================================
import os
import base64
import mimetypes
import time
import math
from openai import OpenAI
from google.colab import drive
from google.colab import userdata
import glob
import json
import concurrent.futures
import pandas as pd
import numpy as np
import time
import base64
import os
from openai import OpenAI
from tqdm import tqdm
# 2. OpenAI 클라이언트 설정
try:
    my_api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=my_api_key)
    print("✅ OpenAI API 키 설정 성공.")
except Exception as e:
    print("🚨 OpenAI API 키 설정 실패: Colab의 'Secrets'에 OPENAI_API_KEY를 설정했는지 확인해주세요.")

print("\n--- 기본 설정 완료 ---")


✅ OpenAI API 키 설정 성공.

--- 기본 설정 완료 ---


In [10]:
# --- ⚙️ 분석 환경 설정 (여기를 수정하세요!) ---

# 1. 분석할 단계별 폴더가 들어있는 기본 경로
BASE_FOLDER_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB"
VALID_EXT = ["jpg", "jpeg", "png"]

# GPT 분석 스키마
JSON_FIELDS = [
    "subfolder1", "subfolder2", "filename",
    "checkerboard_presence",       # yes / partial / no
    "checkerboard_angle_deg",      # int
    "scaling_feasibility",         # yes / borderline / no
    "note"
]

SYSTEM_PROMPT = """
너는 양상추 수직농장 이미지 품질 평가 전문가이다.

가장 중요한 임무는 '체커보드를 반드시 찾아내는 것'이다.
체커보드는 보통 이미지의 매우 아래쪽, 좌측 하단 또는 우측 하단 구석에 작게 존재한다.
양상추가 화면 대부분을 차지하더라도 체커보드 탐지를 최우선으로 수행해야 한다.

체커보드가 매우 작게 보이거나, 부분적으로 잘리거나, 일부만 보이더라도
흑백 격자 패턴(Checkerboard-like grid)가 조금이라도 감지되면 반드시
"yes" 또는 "partial"로 판단하라.

정말로 완전히 보이지 않을 때만 "no"로 판단하라.

추정해야 할 주요 항목:
1) 체커보드 존재 여부 (yes / partial / no)
2) 체커보드 기울기 각도 (정수 deg)
3) 체커보드 기반 스케일 추정 가능 여부 (yes / borderline / no)

출력은 반드시 JSON만 사용한다.
JSON 외 텍스트 출력 금지.
체커보드가 없으면 angle = 0, scaling_feasibility = "no" 로 고정한다.
"""

# -------------------------------------------------------------
# 1️⃣ GPT 분석 함수 (이미지 1장)
# -------------------------------------------------------------
def analyze_one_image(item):
    """
    item: (img_path, sub1, sub2, filename)
    """
    img_path, sub1, sub2, filename = item

    try:
        with open(img_path, "rb") as f:
            image_bytes = f.read()
        image_b64 = base64.b64encode(image_bytes).decode("utf-8")

        user_prompt = f"""
이 이미지는 cam0/cam1의 경우 체커보드가 대부분 '아래쪽 구석'에 작게 존재한다.
체커보드가 아주 작은 영역만 보이거나 부분적으로 잘려 있어도 반드시 우선적으로 탐지하라.

반드시 아래 항목만 JSON으로 출력하라:
- subfolder1: "{sub1}"
- subfolder2: "{sub2}"
- filename: "{filename}"
- checkerboard_presence: "yes" 또는 "partial" 또는 "no"
- checkerboard_angle_deg: 정수
- scaling_feasibility: "yes" 또는 "borderline" 또는 "no"
- note: 체커보드 인식 관련 간단 코멘트

주의:
1) 체커보드를 최우선으로 찾는다.
2) 화면 하단·모서리·작은 패턴을 특히 집중해서 판단한다.
3) JSON 외 텍스트 절대 금지.
"""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
                {
                    "role": "user",
                    "content": [{
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{image_b64}"
                    }]
                },
            ]
        )

        text = response.choices[0].message.content

        try:
            parsed = json.loads(text)
        except:
            # GPT 출력 오류 → 빈 값 넣기
            parsed = {k: None for k in JSON_FIELDS}

        return parsed

    except Exception as e:
        # 파일 로딩 문제 → 빈값 기록
        return {
            "subfolder1": sub1,
            "subfolder2": sub2,
            "filename": filename,
            "checkerboard_presence": None,
            "checkerboard_angle_deg": None,
            "scaling_feasibility": None,
            "note": f"Error: {e}"
        }



# -------------------------------------------------------------
# 2️⃣ 폴더 전체 탐색 (glob)
# -------------------------------------------------------------
def collect_image_paths(base_path):
    all_paths = []
    pattern = os.path.join(base_path, "**", "*.*")
    for path in glob.glob(pattern, recursive=True):
        if path.split(".")[-1].lower() in VALID_EXT:
            # subfolder1 / subfolder2 추출
            parts = path.replace(base_path, "").strip("/").split("/")
            if len(parts) >= 3:
                sub1, sub2, filename = parts[-3], parts[-2], parts[-1]
            else:
                sub1, sub2, filename = ("", "", os.path.basename(path))
            all_paths.append((path, sub1, sub2, filename))
    return all_paths



# -------------------------------------------------------------
# 3️⃣ 병렬 처리 + 진행률 + 남은시간
# -------------------------------------------------------------
def run_parallel_analysis(base_path, max_workers=4):
    items = collect_image_paths(base_path)
    total = len(items)

    print(f"\n📌 분석할 이미지 수: {total}")

    results = []
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(analyze_one_image, item) for item in items]

        for i, f in enumerate(tqdm(concurrent.futures.as_completed(futures), total=total)):
            res = f.result()
            results.append(res)

            elapsed = time.time() - start
            rate = (i + 1) / elapsed
            remaining = (total - (i + 1)) / rate if rate > 0 else 999

            if (i + 1) % 10 == 0:
                print(f"진행 {i+1}/{total}  |  남은시간 ≈ {remaining/60:.1f}분")

    df = pd.DataFrame(results)
    return df

In [12]:

# -------------------------------------------------------------
# 4️⃣ 전체 실행 + 엑셀 저장
# -------------------------------------------------------------
def generate_excel(base_path, output_excel="/content/drive/MyDrive/양상추 분류모델/Image 분석DF/251128_checkerboard_analysis.xlsx", workers=4):
    df = run_parallel_analysis(base_path, max_workers=workers)

    # 컬럼 순서 정렬
    for col in JSON_FIELDS:
        if col not in df.columns:
            df[col] = None

    df = df[JSON_FIELDS]

    df.to_excel(output_excel, index=False)
    print(f"\n📁 엑셀 저장 완료: {output_excel}")

    return df
def create_summary_report(df):
    """
    df: generate_excel()로 만든 DataFrame
    """

    summary_prompt = f"""
너는 양상추 이미지 품질 진단 전문가이다.

아래는 이미 분석된 DataFrame 일부이다:

{df.head(20).to_string()}

전체 DataFrame을 기반으로 종합 보고서를 작성하라.

📌 포함해야 하는 내용:
1) 체커보드 없는 이미지 목록 요약
2) 스케일 추정 불가능 이미지 요약
3) cam2(정면)는 체커보드 거의 없는 것이 정상이라는 점 구분해 서술
4) 체커보드 각도 패턴을 기반으로 ROI 추출 권장 전략 제시
5) 반복적 문제(특정 날짜, 특정 카메라에서 계속 나타나는 패턴)

형식: 순수 텍스트 보고서
표 금지.
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "너는 농업영상 품질 진단 전문가이다."},
            {"role": "user", "content": summary_prompt},
        ],
        temperature=0
    )

    report = response.choices[0].message.content
    return report


In [13]:
df = generate_excel(BASE_FOLDER_PATH, output_excel="/content/drive/MyDrive/양상추 분류모델/Image 분석DF/251128_checkerboard_analysis1.xlsx", workers=4)




📌 분석할 이미지 수: 398


  3%|▎         | 11/398 [00:01<00:39,  9.84it/s]

진행 10/398  |  남은시간 ≈ 0.8분


  5%|▌         | 21/398 [00:02<00:35, 10.65it/s]

진행 20/398  |  남은시간 ≈ 0.7분


...

진행 380/398  |  남은시간 ≈ 0.0분


 98%|█████████▊| 391/398 [00:37<00:00, 10.41it/s]

진행 390/398  |  남은시간 ≈ 0.0분


100%|██████████| 398/398 [00:38<00:00, 10.38it/s]


📁 엑셀 저장 완료: /content/drive/MyDrive/양상추 분류모델/Image 분석DF/251128_checkerboard_analysis1.xlsx


In [14]:
report = create_summary_report(df)
print(report)

### 양상추 이미지 품질 진단 종합 보고서

#### 1) 체커보드 없는 이미지 목록 요약
분석된 이미지 중 체커보드가 없는 이미지가 다수 발견되었습니다. 이들 이미지는 품질 진단에 필요한 기준을 충족하지 못하며, 체커보드의 부재는 이미지의 정확한 스케일링 및 분석을 방해합니다. 체커보드가 없는 이미지 목록은 다음과 같습니다:
- bed15_20251127_062320_cam0.jpg
- bed14_20251127_062123_cam1.jpg
- bed15_20251127_062320_cam1.jpg
- bed14_20251127_062123_cam2.jpg
- bed16_20251127_062514_cam0.jpg
- bed16_20251127_062514_cam1.jpg
- bed15_20251127_062320_cam2.jpg
- bed16_20251127_062514_cam2.jpg
- bed17_20251127_062712_cam0.jpg
- bed17_20251127_062712_cam1.jpg
- bed17_20251127_062712_cam2.jpg
- bed19_20251127_062908_cam0.jpg
- bed19_20251127_062908_cam2.jpg
- bed18_20251127_063103_cam2.jpg
- bed19_20251127_062908_cam1.jpg
- bed18_20251127_063103_cam1.jpg
- bed18_20251127_063103_cam0.jpg
- bed20_20251127_063303_cam1.jpg
- bed20_20251127_063303_cam2.jpg
- bed20_20251127_063303_cam0.jpg

#### 2) 스케일 추정 불가능 이미지 요약
모든 이미지에서 스케일 추정이 불가능한 것으로 나타났습니다. 이는 체커보드의 부재와 관련이 있으며, 스케일링이 필요한 분석을 수행할 수 없음을 의미합니다. 스케일 추정이 불가능한 이미지는 품질 진단 및 후속 분석에 큰 제약을 초래